# Tutorial for the PwInput class

The `PwInput` class creates, reads, modifies and writes the input files of the `pw.x` code of QuantumESPRESSO.
The class inherits from `dict`: the namelists (`control`, `system`, `electrons`, `ions`, `cell`) and the cards
(`atomic_species`, `atomic_positions`, `kpoints`, `cell_parameters`) are keys of the object, so the input can be
edited both with the usual dictionary syntax and with the set methods of the class.

The documentation of each method can be accessed in the notebook as `inp.set_kpoints?`.

In [1]:
# useful to autoreload the module without restarting the kernel
%load_ext autoreload
%autoreload 2

import os, datetime
from mppi import InputFiles as I
from mppi.Calculators.Tools import build_pw_kpath

print('Notebook executed on', datetime.datetime.now().strftime('%d/%m/%Y %H:%M'))

Notebook executed on 30/09/2026 14:32


## Build an input from scratch

An empty object contains only some default values, defined in the `default` member of the class

In [2]:
inp = I.PwInput()
inp

{'control': {'calculation': "'scf'",
  'verbosity': "'high'",
  'prefix': "'pwscf'",
  'outdir': "'./'"},
 'system': {'force_symmorphic': '.false.'},
 'electrons': {'diago_full_acc': '.false.'},
 'ions': {},
 'cell': {},
 'atomic_species': {},
 'atomic_positions': {},
 'kpoints': {},
 'cell_parameters': {}}

The set methods add the variables needed by a computation. As an example we build the input of a scf computation
for bulk silicon

In [3]:
inp.set_scf(conv_thr=1e-8)
inp.set_prefix('si_scf')
inp.set_pseudo_dir(pseudo_dir='../pseudos')
inp.set_energy_cutoff(30)
inp.set_lattice(ibrav=2,celldm1=10.3)
# add the atomic species, then set the number of atoms in the cell (ntyp is computed from the added species)
inp.add_atom('Si','Si.pbe-mt_fhi.UPF')
inp.set_atoms_number(2)
inp.set_atomic_positions([['Si',[0.,0.,0.]],['Si',[0.25,0.25,0.25]]],type='alat')
inp.set_kpoints(type='automatic',points=[4,4,4])
print(inp.convert_string())

&control
         calculation = 'scf'
              outdir = './'
              prefix = 'si_scf'
          pseudo_dir = '../pseudos'
           verbosity = 'high'
/&end
&system
           celldm(1) = 10.3
             ecutwfc = 30
    force_symmorphic = .false.
               ibrav = 2
                 nat = 2
                ntyp = 1
/&end
&electrons
            conv_thr = 1e-08
      diago_full_acc = .false.
/&end
ATOMIC_SPECIES
  Si      1.0    Si.pbe-mt_fhi.UPF
ATOMIC_POSITIONS { alat }
 Si   0.0000000000   0.0000000000   0.0000000000
 Si   0.2500000000   0.2500000000   0.2500000000
K_POINTS { automatic }
  4  4  4  0  0  0


Any variable can also be set with the dictionary syntax. Note that the string values must include the quotes
required by the Fortran syntax of the input

In [4]:
inp['system']['nbnd'] = 8
inp['control']['tprnfor'] = '.true.'
inp['system']

{'force_symmorphic': '.false.',
 'ecutwfc': 30,
 'ibrav': 2,
 'celldm(1)': 10.3,
 'ntyp': 1,
 'nat': 2,
 'nbnd': 8}

The input is written on file with the `write` method, and an existing file is read by passing it to the constructor

In [5]:
os.makedirs('PwInput_tutorial',exist_ok=True)
inp.write('PwInput_tutorial/si_scf.in')

inp2 = I.PwInput(file='PwInput_tutorial/si_scf.in')
print('The input read from file is equal to the original one:',
      all(inp2[key] == inp[key] for key in inp.namelist + inp.cards))

The input read from file is equal to the original one: True


## Read and modify an existing input

We read the input of a scf computation for GaAs. All the namelists are parsed, while among the cards the class
manages ATOMIC_SPECIES, ATOMIC_POSITIONS, K_POINTS and CELL_PARAMETERS

In [6]:
inp = I.PwInput(file='IO_files/gaas_scf.in')
inp

{'control': {'calculation': "'scf'",
  'verbosity': "'high'",
  'prefix': "'ecut_40-k_4'",
  'outdir': "'./'",
  'pseudo_dir': "'../pseudos'"},
 'system': {'force_symmorphic': '.false.',
  'occupations': "'fixed'",
  'ibrav': 2,
  'celldm(1)': 10.677,
  'ntyp': 2,
  'nat': 2,
  'ecutwfc': 40},
 'electrons': {'diago_full_acc': '.false.', 'conv_thr': 1e-08},
 'ions': {},
 'cell': {},
 'atomic_species': {'Ga': [69.72, 'Ga_hamlu.fhi.UPF'],
  'As': [74.92, 'As_hamlu.fhi.UPF']},
 'atomic_positions': {'type': 'alat',
  'values': [['Ga', [0.0, 0.0, 0.0]], ['As', [0.25, 0.25, 0.25]]]},
 'kpoints': {'type': 'automatic',
  'values': ([4.0, 4.0, 4.0], [0.0, 0.0, 0.0])},
 'cell_parameters': {},
 'file': 'IO_files/gaas_scf.in'}

and we turn it into the input of a band structure computation along a path of high symmetry points
(in cartesian coordinates, in units of 2$\pi$/alat). The `build_pw_kpath` function builds the list of points in
the format required by the `tpiba_b` option

In [7]:
G, X, L, W = [0.,0.,0.], [1.,0.,0.], [0.5,0.5,0.5], [1.,0.5,0.]
inp.set_bands(nbnd=12)
inp.set_kpoints(type='tpiba_b',klist=build_pw_kpath(G,X,L,W,numstep=30))
print(inp.convert_string())

&control
         calculation = 'bands'
              outdir = './'
              prefix = 'ecut_40-k_4'
          pseudo_dir = '../pseudos'
           verbosity = 'high'
/&end
&system
           celldm(1) = 10.677
             ecutwfc = 40
    force_symmorphic = .false.
               ibrav = 2
                 nat = 2
                nbnd = 12
                ntyp = 2
         occupations = 'fixed'
/&end
&electrons
            conv_thr = 1e-08
      diago_full_acc = .false.
/&end
ATOMIC_SPECIES
  Ga    69.72     Ga_hamlu.fhi.UPF
  As    74.92     As_hamlu.fhi.UPF
ATOMIC_POSITIONS { alat }
 Ga   0.0000000000   0.0000000000   0.0000000000
 As   0.2500000000   0.2500000000   0.2500000000
K_POINTS { tpiba_b }
4
  0.00000000   0.00000000   0.00000000  30.00000000 
  1.00000000   0.00000000   0.00000000  30.00000000 
  0.50000000   0.50000000   0.50000000  30.00000000 
  1.00000000   0.50000000   0.00000000   0.00000000 


## Lattice defined by the cell vectors and constraints on the atoms

If `ibrav=0` the lattice vectors are read from the CELL_PARAMETERS card and stored in the `cell_parameters` key,
as in this input for graphene

In [8]:
inp = I.PwInput(file='IO_files/graphene_nscf.in')
print('prefix :',inp.get_prefix())
inp['cell_parameters']

prefix : ecut:100,k:9


{'type': 'angstrom',
 'values': [[2.4595121467, 0.0, 0.0],
  [1.2297560734, 2.13, 0.0],
  [0.0, 0.0, 10.0]]}

In a relax computation the `if_pos` flags, given as a third element of the atom, fix the components of the atomic
positions (0 = fixed). Here the first Si atom is kept fixed and the second one is displaced

In [9]:
inp = I.PwInput(file='IO_files/si_scf.in')
inp['control']['calculation'] = "'relax'"
inp['ions']['ion_dynamics'] = "'bfgs'"
inp.set_atomic_positions([['Si',[0.,0.,0.],[0,0,0]],['Si',[0.26,0.24,0.25]]],type='crystal')
print(inp.convert_string())

&control
         calculation = 'relax'
              outdir = './'
              prefix = 'si_scf'
          pseudo_dir = '../pseudos'
           verbosity = 'high'
/&end
&system
           celldm(1) = 10.3
             ecutwfc = 40
    force_symmorphic = .true.
               ibrav = 2
                 nat = 2
                ntyp = 1
         occupations = 'fixed'
/&end
&electrons
            conv_thr = 1e-08
      diago_full_acc = .false.
/&end
&ions
        ion_dynamics = 'bfgs'
/&end
ATOMIC_SPECIES
  Si   28.086    Si.pbe-mt_fhi.UPF
ATOMIC_POSITIONS { crystal }
 Si   0.0000000000   0.0000000000   0.0000000000 0 0 0
 Si   0.2600000000   0.2400000000   0.2500000000
K_POINTS { automatic }
  4  4  4  0  0  0
